In [2]:
!apt-get install redis-server
!pip install redis

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-tools
Suggested packages:
  ruby-redis
The following NEW packages will be installed:
  libjemalloc2 liblua5.1-0 liblzf1 lua-bitop lua-cjson redis-server
  redis-tools
0 upgraded, 7 newly installed, 0 to remove and 30 not upgraded.
Need to get 1,273 kB of archives.
After this operation, 5,725 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjemalloc2 amd64 5.2.1-4ubuntu1 [240 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblua5.1-0 amd64 5.1.5-8.1build4 [99.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 liblzf1 amd64 3.6-3 [7,444 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 lua-bitop amd64 1.0.2-5 [6,680 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 lua

STRINGS

In [3]:
# Start Redis server in the background
import os
os.system("redis-server --daemonize yes")

0

In [4]:
!ps -ef | grep redis

root        1535       1  0 21:30 ?        00:00:00 redis-server *:6379
root        1548     922  0 21:30 ?        00:00:00 /bin/bash -c ps -ef | grep redis
root        1550    1548  0 21:30 ?        00:00:00 grep redis


In [5]:
# Implement a simple key-value store. (Basic SET and GET)
import redis

r = redis.Redis(host='localhost',port=6379)

r.set('Name','Poorvi')
r.set('email','poorvika@gmail.com')

print(r.get('Name'))
print(r.get('email'))

b'Poorvi'
b'poorvika@gmail.com'


In [6]:
# Build a website hit counter.
def hitpage(page_id):
    r.incr(f'page: {page_id}: hits')
    return int( r.get(f'page: {page_id}: hits'))

for _ in range(5):
    hitpage('home')
print(f"Home page hits: {hitpage('home')}" )

Home page hits: 6


In [7]:
user_id = 1001
r.hset(f'user:{user_id}', mapping={
'name': 'Bob',
'email': 'bob@example.com',
'age': '30',
'city': 'New York'
})
print(r.hgetall(f'user:{user_id}'))
print(r.hget(f'user:{user_id}', 'name'))

{b'name': b'Bob', b'email': b'bob@example.com', b'age': b'30', b'city': b'New York'}
b'Bob'


In [8]:
# Function to create a new user profile
def create_user(user_id, name, email, age):
    user_key = f"user:{user_id}"
    r.hset(user_key, mapping={
        'name': name,
        'email': email,
        'age': age
    })

# Function to get a user profile
def get_user(user_id):
    user_key = f"user:{user_id}"
    return r.hgetall(user_key)

# Function to update a field in the user profile
def update_user_field(user_id, field, value):
    user_key = f"user:{user_id}"
    r.hset(user_key, field, value)

# Function to delete a user profile
def delete_user(user_id):
    user_key = f"user:{user_id}"
    r.delete(user_key)

# Example usage
create_user("001", "Alice", "alice@example.com", 25)
create_user("002", "Bob", "bob@example.com", 30)

print("User 001:", get_user("001"))
print("User 002:", get_user("002"))

# Update email for Alice
update_user_field("001", "email", "alice@newdomain.com")

# Retrieve and print user profiles
print("\n\nUser 001:", get_user("001"))
print("User 002:", get_user("002"))

# Delete user 002
delete_user("002")
print("User 002 after deletion:", get_user("002"))  # Should be empty

User 001: {b'name': b'Alice', b'email': b'alice@example.com', b'age': b'25'}
User 002: {b'name': b'Bob', b'email': b'bob@example.com', b'age': b'30'}


User 001: {b'name': b'Alice', b'email': b'alice@newdomain.com', b'age': b'25'}
User 002: {b'name': b'Bob', b'email': b'bob@example.com', b'age': b'30'}
User 002 after deletion: {}


In [16]:
r.hset('product:1001',mapping=
       {
           'name':'Laptop',
           'price':'85000',
           'stock':'20',
           'category':'electronics'
       })
r.hset('product:1002',mapping=
      {
            'name':'iphone 17',
            'price':'90000',
            'stock':'200',
            'category':'electronics'
      })

print(r.hgetall('product:1001'))
print(r.hget('product:1001','stock'))
r.hincrby('product:1001','stock',-1)

{b'name': b'Laptop', b'price': b'85000', b'stock': b'20', b'category': b'electronics'}
b'20'


19

LISTS

In [17]:
queue_name = 'msg_que'
r.lpush(queue_name,'msg1')
r.lpush(queue_name,'msg2')
r.lpush(queue_name,'msg3')
r.lpush(queue_name,'msg4')
while True:
  message = r.rpop(queue_name)
  if not message:
    break
  print(f"Received message: {message.decode('utf-8')}")

Received message: msg1
Received message: msg2
Received message: msg3
Received message: msg4


In [26]:
feed_key = 'user:1000:feed'
activities = [
    'Logged in',
    'Viewed profile',
    'Posted update',
    'Liked a post',
    'Commented',
    'Logged out' ]

for act in activities:
  r.lpush(feed_key,act)
  r.ltrim(feed_key,0,4)
  print(r.lrange(feed_key,0,-1))

[b'Logged in', b'Logged out', b'Commented', b'Liked a post', b'Posted update']
[b'Viewed profile', b'Logged in', b'Logged out', b'Commented', b'Liked a post']
[b'Posted update', b'Viewed profile', b'Logged in', b'Logged out', b'Commented']
[b'Liked a post', b'Posted update', b'Viewed profile', b'Logged in', b'Logged out']
[b'Commented', b'Liked a post', b'Posted update', b'Viewed profile', b'Logged in']
[b'Logged out', b'Commented', b'Liked a post', b'Posted update', b'Viewed profile']


SETS

In [28]:
article_id = 1001
r.sadd(f'article:{article_id}:tag',
       'tech','python','programming')
article_id = 1002
r.sadd(f'article:{article_id}:tag',
       'tech','gaming','reviews')

print(r.smembers(f'article:{1001}:tag'))
common_tags = r.sinter('article:1001:tag','article:1002:tag')
print('Common Tags: ',common_tags)

{b'python', b'tech', b'programming'}
Common Tags:  {b'tech'}


In [34]:
r.sadd('user:1000:friends','1001','1002','1003')
r.sadd('user:1001:friends','1000','1002', '1004')
r.sadd('user:1002:friends', '1000', '1001', '1005')

mutual_friends = r.sinter('user:1000:friends','user:1001:friends')
print("Mutual Friend of 1000 and 1001: ",mutual_friends)

temp_key = 'temp:fof'
r.sunionstore(temp_key, 'user:1001:friends','user:1002:friends')

recomm = r.sdiff(temp_key,'user:1000:friends')
r.delete(temp_key)
print("Recommended Friends: ",recomm)

Mutual Friend of 1000 and 1001:  {b'1002'}
Recommended Friends:  {b'1000', b'1004', b'1005'}


SORTED SETS

In [115]:
leaderboard = 'game_lb'
r.zadd(leaderboard,{
    'p1': 1500,
    'p2': 1800,
    'p3': 1200,
    'p4': 2000,
    'p5': 1700
})

r.zincrby(leaderboard,300,'p3')
top_players = r.zrevrange(leaderboard,0,2,withscores=True)
print("Top 3 Players: \n",top_players)

for rank,(player,score) in enumerate(top_players,start=1):
  print(f"{rank}. {player.decode('utf-8')} - {score}")

print(f"Player 3 rank: {r.zrevrank(leaderboard,'p3')+1}")

Top 3 Players: 
 [(b'p4', 2000.0), (b'p2', 1800.0), (b'p5', 1700.0)]
1. p4 - 2000.0
2. p2 - 1800.0
3. p5 - 1700.0
Player 3 rank: 4


In [43]:
import time
event_que = 'time_events'
current_time = int(time.time())

r.zadd(event_que,{
    'event1': current_time + 10,
    'event2': current_time + 30,
    'event3': current_time + 5,
    'event4': current_time + 60
    })

ready_events = r.zrangebyscore(event_que,0,current_time+15)
print(f"Events ready to process:{ready_events}")


Events ready to process:[b'event3', b'event1']


HASHES EXERCISE

In [47]:
def create_user_profile(user_id,name,age,email):
  r.hset(f'user:{user_id}',mapping=
         {
             'name':name,
             'age':age,
             'email':email
         })
  print(f"Created profile for user {user_id}")

create_user_profile(10,'Poorvi','19','poorvik@gmail.com')
print(r.hgetall('user:10'))

def get_user_field(user_id,field):
  value = r.hget(f'user:{user_id}',field)
  return value.decode('utf-8') if value else None

print(get_user_field(1001, 'name'))
print(get_user_field(1001, 'age'))

Created profile for user 10
{b'name': b'Poorvi', b'age': b'19', b'email': b'poorvik@gmail.com'}
Bob
30


In [59]:
def dic_redisHash(key,data_dict):
  r.hset(key,mapping=data_dict)

user_data = {
      'username': 'Poorvika',
      'full_name': 'Poorvika Gowda',
      'joined_on': '2025-05-15' }

dic_redisHash('user:1000',user_data)
print("Dictinoary stored has Redis Hash:\n",r.hgetall('user:1000'))

def redis_dict(key):
  return {k.decode('utf-8'):v.decode('utf-8') for k,v in r.hgetall(key).items()}
  # for k,v in r.hgetall(key).items():
  #   return {k.decode('utf-8'):v.decode('utf-8')}

print("Redis hash to dict:\n",redis_dict('user:1000'))

Dictinoary stored has Redis Hash:
 {b'username': b'Poorvika', b'full_name': b'Poorvika Gowda', b'joined_on': b'2025-05-15'}
Redis hash to dict:
 None


LISTS AND QUEUES

In [74]:
def enqueue_msg(qname,msg):
  return r.lpush(qname,msg)
def dequeue_msg(qname):
  return r.rpop(qname)

enqueue_msg('msg_que','msg1')
enqueue_msg('msg_que','msg2')
enqueue_msg('msg_que','msg3')
enqueue_msg('msg_que','msg4')

print(dequeue_msg('msg_que'))


b'msg1'


In [76]:
def log_activity(user_id,activity):
  r.lpush(f'user:{user_id}:activity',activity)
  r.ltrim(f'user:{user_id}:activity',0,9)

for i in range(15):
  log_activity(1001,f"Action{i}")

print(r.lrange(f'user:{1001}:activity',0,-1))

[b'Action14', b'Action13', b'Action12', b'Action11', b'Action10', b'Action9', b'Action8', b'Action7', b'Action6', b'Action5']


In [80]:
def remove_list_duplicates(key):
  elements = [item.decode('utf-8') for item in r.lrange(key, 0, -1)]
  unique_elements = list(dict.fromkeys(elements))
  r.delete(key)
  if unique_elements:
    r.rpush(key, *unique_elements)
r.rpush('dupe_list', 'a', 'b', 'a', 'c', 'b', 'd')
remove_list_duplicates('dupe_list')
print(r.lrange('dupe_list', 0, -1))

[b'a', b'b', b'c', b'd']


SETS AND SETS OPERATIONS

In [86]:
def add_article_tag(article_id,*tag):
  r.sadd(f'article:{article_id}:tags',*tag)

def get_article_tags(article_id):
  return r.smembers(f'article:{article_id}:tags')

add_article_tag(1001,'tech','python','programming')
print(get_article_tags(1001))

{b'python', b'tech', b'programming'}


In [111]:

def recommend_friends(user):
    user_friends = r.smembers(f'user:{user}')
    recommendations = set()

    for friend in user_friends:
        friend = friend.decode('utf-8')
        mutuals = r.sinter(f'user:{user}', f'user:{friend}')
        mutuals = {m.decode('utf-8') for m in mutuals}

        # Get this friend's friends
        friend_friends = r.smembers(f'user:{friend}')
        friend_friends = {f.decode('utf-8') for f in friend_friends}

        # Recommend friends-of-friends who are not already friends
        potential = friend_friends - {user} - {f.decode('utf-8') for f in user_friends}
        recommendations.update(potential)

    return list(recommendations)

r.sadd('user:alice','bob','charlie','dava')
r.sadd('user:bob','alice','dava','emma')
r.sadd('user:charlie','alice','emma','frank')
recommend_friends('alice')

['frank', 'elice', 'emma']